# Exploratory Data Analysis — Titanic Dataset

This notebook performs **Univariate, Bivariate, and Multivariate Analysis** on the final preprocessed dataset:

```text
06_titanic_feature_engineered.csv
```

The dataset has already gone through:

```text
Missing Values → Outliers → Encoding → Scaling → Feature Engineering
```

EDA is used to understand distributions, relationships, patterns, and possible data-quality issues before Machine Learning.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## 2. Load the Final Dataset

If this notebook is inside:

```text
04-Exploratory-Data-Analysis/
    All-Analysis.ipynb
```

and your dataset is inside the project `Dataset` folder, use the relative path below.

In [ ]:
df = pd.read_csv("../03-Data-preprocessing/Dataset/06_titanic_feature_engineered.csv")

df.head()

## 3. Basic Dataset Understanding

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
missing = df.isnull().sum()
display(missing[missing > 0])

print("\nTarget distribution:")
display(df["Survived"].value_counts())

# Univariate Analysis

Univariate analysis studies **one variable at a time**.

## 4. Numerical Summary

In [ ]:
numerical_features = [
    "Age", "Fare", "Pclass", "SibSp", "Parch",
    "FamilySize", "TicketGroupSize", "Log_Fare", "Sqrt_Fare"
]

df[numerical_features].describe()

## 5. Distribution of Age

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["Age"].dropna(), bins=30)
plt.title("Distribution of Age")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 6. Distribution of Fare

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["Fare"].dropna(), bins=30)
plt.title("Distribution of Fare")
plt.xlabel("Fare")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 7. Distribution of Engineered Numerical Features

In [ ]:
for feature in ["FamilySize", "TicketGroupSize"]:
    plt.figure(figsize=(8, 4))
    plt.hist(df[feature].dropna(), bins=20)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

## 8. Box Plots

In [ ]:
for feature in numerical_features:
    plt.figure(figsize=(8, 3))
    plt.boxplot(df[feature].dropna(), vert=False)
    plt.title(f"Box Plot of {feature}")
    plt.xlabel(feature)
    plt.tight_layout()
    plt.show()

## 9. Binary / Categorical Frequencies

In [ ]:
binary_features = [
    "Sex_male", "Embarked_Q", "Embarked_S", "IsAlone"
]

for feature in binary_features:
    print(f"--- {feature} ---")
    display(df[feature].value_counts().sort_index())

## 10. Title and Age-Group Features

In [ ]:
title_features = [col for col in df.columns if col.startswith("Title_")]
age_group_features = [col for col in df.columns if col.startswith("Age_Group_")]

print("Title frequencies:")
display(df[title_features].sum().sort_values(ascending=False))

print("\nAge-group frequencies:")
display(df[age_group_features].sum().sort_values(ascending=False))

# Bivariate Analysis

Bivariate analysis studies the relationship between **two variables**.

The main question here is:

> How does each important feature relate to `Survived`?

## 11. Survival Distribution

In [ ]:
survival_counts = df["Survived"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Did Not Survive", "Survived"], survival_counts.values)
plt.title("Survival Distribution")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.show()

## 12. Pclass vs Survival

In [ ]:
pclass_survival = df.groupby("Pclass")["Survived"].mean()

display(pclass_survival.to_frame("Survival Rate"))

plt.figure(figsize=(6, 4))
plt.bar(pclass_survival.index.astype(str), pclass_survival.values)
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Pclass")
plt.ylabel("Survival Rate")
plt.tight_layout()
plt.show()

## 13. Sex vs Survival

In [ ]:
sex_survival = df.groupby("Sex_male")["Survived"].mean()

display(sex_survival.to_frame("Survival Rate"))

plt.figure(figsize=(6, 4))
plt.bar(["Female", "Male"], sex_survival.values)
plt.title("Survival Rate by Sex")
plt.ylabel("Survival Rate")
plt.tight_layout()
plt.show()

## 14. IsAlone vs Survival

In [ ]:
alone_survival = df.groupby("IsAlone")["Survived"].mean()

display(alone_survival.to_frame("Survival Rate"))

plt.figure(figsize=(6, 4))
plt.bar(["Not Alone", "Alone"], alone_survival.values)
plt.title("Survival Rate by Travel Status")
plt.ylabel("Survival Rate")
plt.tight_layout()
plt.show()

## 15. FamilySize vs Survival

In [ ]:
family_survival = df.groupby("FamilySize")["Survived"].mean().sort_index()

display(family_survival.to_frame("Survival Rate"))

plt.figure(figsize=(8, 4))
plt.plot(family_survival.index, family_survival.values, marker="o")
plt.title("Survival Rate by Family Size")
plt.xlabel("Family Size")
plt.ylabel("Survival Rate")
plt.tight_layout()
plt.show()

## 16. Age vs Survival

In [ ]:
age_survival = df.groupby("Survived")["Age"].agg(["mean", "median", "min", "max"])
age_survival.index = ["Did Not Survive", "Survived"]
age_survival

## 17. Fare vs Survival

In [ ]:
fare_survival = df.groupby("Survived")["Fare"].agg(["mean", "median", "min", "max"])
fare_survival.index = ["Did Not Survive", "Survived"]
fare_survival

## 18. Embarked vs Survival

In [ ]:
for feature in ["Embarked_Q", "Embarked_S"]:
    result = df.groupby(feature)["Survived"].mean()
    print(f"--- {feature} ---")
    display(result.to_frame("Survival Rate"))

## 19. Ticket Group Size vs Survival

In [ ]:
ticket_group_survival = (
    df.groupby("TicketGroupSize")["Survived"]
      .agg(["mean", "count"])
      .rename(columns={"mean": "Survival Rate", "count": "Passengers"})
)

ticket_group_survival.sort_index()

# Multivariate Analysis

Multivariate analysis studies **multiple variables together**.

## 20. Correlation Matrix

In [ ]:
correlation = df.select_dtypes(include="number").corr()

plt.figure(figsize=(14, 10))
plt.imshow(correlation, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=90)
plt.yticks(range(len(correlation.columns)), correlation.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 21. Correlation with the Target

In [ ]:
target_correlation = (
    df.select_dtypes(include="number")
      .corr()["Survived"]
      .drop("Survived")
      .sort_values(key=abs, ascending=False)
)

target_correlation.to_frame("Correlation with Survived")

## 22. Sex + Pclass vs Survival

In [ ]:
sex_class_survival = (
    df.groupby(["Sex_male", "Pclass"])["Survived"]
      .mean()
      .reset_index()
)

sex_class_survival["Sex"] = sex_class_survival["Sex_male"].map({
    0: "Female",
    1: "Male"
})

sex_class_survival[["Sex", "Pclass", "Survived"]].rename(
    columns={"Survived": "Survival Rate"}
)

## 23. FamilySize + Sex vs Survival

In [ ]:
family_sex_survival = (
    df.groupby(["FamilySize", "Sex_male"])["Survived"]
      .mean()
      .reset_index()
)

family_sex_survival["Sex"] = family_sex_survival["Sex_male"].map({
    0: "Female",
    1: "Male"
})

family_sex_survival[["FamilySize", "Sex", "Survived"]].rename(
    columns={"Survived": "Survival Rate"}
)

## 24. Pclass + IsAlone vs Survival

In [ ]:
class_alone_survival = (
    df.groupby(["Pclass", "IsAlone"])["Survived"]
      .mean()
      .reset_index()
)

class_alone_survival["Travel Status"] = class_alone_survival["IsAlone"].map({
    0: "Not Alone",
    1: "Alone"
})

class_alone_survival[[
    "Pclass", "Travel Status", "Survived"
]].rename(columns={"Survived": "Survival Rate"})

## 25. Age + Sex + Survival

In [ ]:
age_sex_summary = (
    df.groupby(["Sex_male", "Survived"])["Age"]
      .agg(["mean", "median", "count"])
      .reset_index()
)

age_sex_summary["Sex"] = age_sex_summary["Sex_male"].map({
    0: "Female",
    1: "Male"
})

age_sex_summary

# 26. Key Findings

Use the analyses above to record the patterns you actually observe.

```text
## Key Findings

1. Distribution
   - Describe important numerical distributions.
   - Note skewed features.

2. Target Relationships
   - Which features show noticeable survival differences?

3. Multivariate Patterns
   - Which combinations of features show different survival rates?

4. Data Quality
   - Were any missing values or unusual distributions found?

5. Modeling Relevance
   - Which features appear worth investigating in the ML phase?
```

## Conclusion

EDA moves the project from:

```text
Final Preprocessed Dataset
          ↓
Understand Distributions
          ↓
Understand Relationships
          ↓
Find Patterns
          ↓
Identify Useful Features
          ↓
Machine Learning
```